# CatBoost/XGBoost 멀티시드 배깅 + 3종 최종 앙상블

LightGBM에서 효과를 본(+0.00038) 멀티시드 배깅을 CatBoost, XGBoost에도
똑같이 적용합니다. 그다음 배깅된 3개 모델을 다시 앙상블해서 최종 후보
제출 파일을 만듭니다.

**XGBoost 카테고리 인코딩 — 대회 규칙 준수**

**train 카테고리만 사용**하도록 고쳤고, test에만 있는 새 값은 자동으로 NaN 처리됩니다
(XGBoost는 결측치를 학습 시점에 정한 기본 분기 방향으로 안전하게 처리하므로 문제없음).
LightGBM(이 노트북이 의존하는 `lgbm_multiseed_bagging.ipynb`)과 CatBoost(아래 3번 셀)는
원래부터 train만 사용하는 안전한 방식이라 그대로 둡니다.

**예상 시간**: CatBoost(10seed×5fold)와 XGBoost(10seed×5fold) 합쳐서
**꽤 오래 걸립니다(1~2시간 예상)**. 시드 수를 줄이고 싶으면 2번 셀의
`SEEDS` 리스트 길이를 줄이세요(예: 5개만 써도 효과의 대부분은 얻을 수 있어요).

**사전 준비**: `catboost_best_params.json`, `xgboost_best_params.json`,
`blend_cache/oof_10seed_bagged.npy`가 있어야 합니다.

## 1. 라이브러리, 설정

In [5]:
import os
import json
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import warnings

warnings.filterwarnings("ignore")

DATA_DIR = "../../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
N_SPLITS = 5
CACHE_DIR = "../blend_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

SEEDS = [1004, 42, 7, 123, 2024, 88, 555, 999, 31, 77]
print(f"시드 {len(SEEDS)}개 사용 (줄이고 싶으면 이 리스트를 줄이세요)")

with open("catboost_best_params.json", "r", encoding="utf-8") as f:
    cat_best_params = json.load(f)
with open("xgboost_best_params.json", "r", encoding="utf-8") as f:
    xgb_best_params = json.load(f)

print("CatBoost params:", cat_best_params)
print("XGBoost params:", xgb_best_params)

시드 10개 사용 (줄이고 싶으면 이 리스트를 줄이세요)
CatBoost params: {'depth': 7, 'learning_rate': 0.027889857017869574, 'l2_leaf_reg': 8.4028044371733, 'bagging_temperature': 0.1838885820165164, 'random_strength': 7.109677945620526, 'border_count': 122}
XGBoost params: {'max_depth': 4, 'learning_rate': 0.022609899133612964, 'subsample': 0.8073464819105185, 'colsample_bytree': 0.9801839769004944, 'min_child_weight': 3, 'reg_alpha': 8.133840106413295, 'reg_lambda': 6.077476819566608, 'gamma': 2.0490086584362506}


## 2. 공통 전처리 (base_features, fill_na)

In [6]:
import os
print("현재 작업 디렉토리:", os.getcwd())
print("../../data 존재?:", os.path.exists("../../data"))
print("../../data/train.csv 존재?:", os.path.exists("../../data/train.csv"))

현재 작업 디렉토리: /Users/sojung/Desktop/AI_Health_care/17. 난임 환자 심신 성공 여부 예측 AI 해커톤/git/infertility_classification/experiment_history/1차
../../data 존재?: True
../../data/train.csv 존재?: True


In [7]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])
test_raw = pd.read_csv(TEST_PATH)
test_ids = test_raw["ID"].copy()
test_raw = test_raw.drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df_train_base = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
df_test_base = fill_na(base_features(test_raw.copy()).drop(columns=DEAD_COLS))

y = df_train_base[TARGET_COL].values.astype(np.float32)
print(f"전처리 완료: train {df_train_base.shape}, test {df_test_base.shape}")

전처리 완료: train (256351, 68), test (90067, 67)


## 3. CatBoost 멀티시드 배깅 (문자열 범주형 그대로)

In [8]:
X_cat_train = df_train_base.drop(columns=[TARGET_COL]).copy()
X_cat_test = df_test_base.copy()

cat_cols = X_cat_train.select_dtypes(include=["object"]).columns.tolist()
for col in cat_cols:
    X_cat_train[col] = X_cat_train[col].astype(str)
    X_cat_test[col] = X_cat_test[col].astype(str)
X_cat_test = X_cat_test[X_cat_train.columns]

cat_oof_per_seed = []
cat_test_per_seed = []

for seed_i, seed in enumerate(SEEDS, start=1):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof_seed = np.zeros(len(y))
    test_pred_seed = np.zeros(len(X_cat_test))

    for tr_idx, val_idx in skf.split(X_cat_train, y):
        model = CatBoostClassifier(
            **cat_best_params,
            cat_features=cat_cols,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
        )
        model.fit(X_cat_train.iloc[tr_idx], y[tr_idx])
        oof_seed[val_idx] = model.predict_proba(X_cat_train.iloc[val_idx])[:, 1]
        test_pred_seed += model.predict_proba(X_cat_test)[:, 1] / N_SPLITS

    seed_auc = roc_auc_score(y, oof_seed)
    cat_oof_per_seed.append(oof_seed)
    cat_test_per_seed.append(test_pred_seed)
    print(f"[CatBoost] 시드 {seed_i}/{len(SEEDS)} (random_state={seed})  OOF AUC: {seed_auc:.5f}")

cat_oof_per_seed = np.array(cat_oof_per_seed)
cat_test_per_seed = np.array(cat_test_per_seed)

oof_catboost_bagged = cat_oof_per_seed.mean(axis=0)
test_catboost_bagged = cat_test_per_seed.mean(axis=0)
score_catboost_bagged = roc_auc_score(y, oof_catboost_bagged)
print(f"\nCatBoost 배깅 OOF AUC: {score_catboost_bagged:.5f}")

np.save(f"{CACHE_DIR}/oof_catboost_bagged.npy", oof_catboost_bagged)
np.save(f"{CACHE_DIR}/test_catboost_bagged.npy", test_catboost_bagged)
print("저장 완료: oof_catboost_bagged.npy, test_catboost_bagged.npy")

[CatBoost] 시드 1/10 (random_state=1004)  OOF AUC: 0.73944
[CatBoost] 시드 2/10 (random_state=42)  OOF AUC: 0.73984
[CatBoost] 시드 3/10 (random_state=7)  OOF AUC: 0.73980
[CatBoost] 시드 4/10 (random_state=123)  OOF AUC: 0.73975
[CatBoost] 시드 5/10 (random_state=2024)  OOF AUC: 0.73980
[CatBoost] 시드 6/10 (random_state=88)  OOF AUC: 0.73977
[CatBoost] 시드 7/10 (random_state=555)  OOF AUC: 0.73975
[CatBoost] 시드 8/10 (random_state=999)  OOF AUC: 0.73978
[CatBoost] 시드 9/10 (random_state=31)  OOF AUC: 0.73982
[CatBoost] 시드 10/10 (random_state=77)  OOF AUC: 0.73979

CatBoost 배깅 OOF AUC: 0.74005
저장 완료: oof_catboost_bagged.npy, test_catboost_bagged.npy


## 4. XGBoost 멀티시드 배깅 (네이티브 범주형) — [수정됨: train 카테고리만 사용]

In [9]:
X_xgb_train = df_train_base.drop(columns=[TARGET_COL]).copy()
X_xgb_test = df_test_base.copy()

xgb_cat_cols = X_xgb_train.select_dtypes(include=["object"]).columns.tolist()
for col in xgb_cat_cols:
    # ★ 대회 규칙 준수: 카테고리 목록은 train에서만 만듦 (test 값은 절대 사용 안 함)
    # 원래 코드는 train ∪ test로 combined_categories를 만들었는데, 이건 DACON 규칙
    # "label encoding 시 test 데이터셋 활용"에 해당하는 Data Leakage라 수정함
    train_categories = sorted(X_xgb_train[col].astype(str).unique())
    X_xgb_train[col] = pd.Categorical(X_xgb_train[col].astype(str), categories=train_categories)
    # test에만 있는 새 값은 train 카테고리에 없으므로 자동으로 NaN이 됨
    # (XGBoost가 학습 시점에 정한 기본 분기 방향으로 결측치를 안전하게 처리)
    X_xgb_test[col] = pd.Categorical(X_xgb_test[col].astype(str), categories=train_categories)
X_xgb_test = X_xgb_test[X_xgb_train.columns]

# 참고용: test에서 NaN으로 바뀐 값이 얼마나 되는지 확인 (너무 많으면 다른 문제가 있다는 신호)
print("train에 없어서 NaN 처리된 test 값 개수 (컬럼별):")
any_na = False
for col in xgb_cat_cols:
    n_na = X_xgb_test[col].isna().sum()
    if n_na > 0:
        any_na = True
        print(f"  {col}: {n_na}개 행")
if not any_na:
    print("  없음 (모든 test 카테고리가 train에 이미 존재)")
print()

xgb_oof_per_seed = []
xgb_test_per_seed = []

for seed_i, seed in enumerate(SEEDS, start=1):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof_seed = np.zeros(len(y))
    test_pred_seed = np.zeros(len(X_xgb_test))

    for tr_idx, val_idx in skf.split(X_xgb_train, y):
        model = xgb.XGBClassifier(
            n_estimators=3000,
            **xgb_best_params,
            tree_method="hist",
            enable_categorical=True,
            eval_metric="auc",
            random_state=seed,
            n_jobs=-1,
            early_stopping_rounds=100,
        )
        model.fit(
            X_xgb_train.iloc[tr_idx], y[tr_idx],
            eval_set=[(X_xgb_train.iloc[val_idx], y[val_idx])],
            verbose=False,
        )
        oof_seed[val_idx] = model.predict_proba(X_xgb_train.iloc[val_idx])[:, 1]
        test_pred_seed += model.predict_proba(X_xgb_test)[:, 1] / N_SPLITS

    seed_auc = roc_auc_score(y, oof_seed)
    xgb_oof_per_seed.append(oof_seed)
    xgb_test_per_seed.append(test_pred_seed)
    print(f"[XGBoost] 시드 {seed_i}/{len(SEEDS)} (random_state={seed})  OOF AUC: {seed_auc:.5f}")

xgb_oof_per_seed = np.array(xgb_oof_per_seed)
xgb_test_per_seed = np.array(xgb_test_per_seed)

oof_xgboost_bagged = xgb_oof_per_seed.mean(axis=0)
test_xgboost_bagged = xgb_test_per_seed.mean(axis=0)
score_xgboost_bagged = roc_auc_score(y, oof_xgboost_bagged)
print(f"\nXGBoost 배깅 OOF AUC: {score_xgboost_bagged:.5f}")

np.save(f"{CACHE_DIR}/oof_xgboost_bagged.npy", oof_xgboost_bagged)
np.save(f"{CACHE_DIR}/test_xgboost_bagged.npy", test_xgboost_bagged)
print("저장 완료: oof_xgboost_bagged.npy, test_xgboost_bagged.npy")

train에 없어서 NaN 처리된 test 값 개수 (컬럼별):
  특정 시술 유형: 3개 행

[XGBoost] 시드 1/10 (random_state=1004)  OOF AUC: 0.73972
[XGBoost] 시드 2/10 (random_state=42)  OOF AUC: 0.73971
[XGBoost] 시드 3/10 (random_state=7)  OOF AUC: 0.73971
[XGBoost] 시드 4/10 (random_state=123)  OOF AUC: 0.73978
[XGBoost] 시드 5/10 (random_state=2024)  OOF AUC: 0.73996
[XGBoost] 시드 6/10 (random_state=88)  OOF AUC: 0.73987
[XGBoost] 시드 7/10 (random_state=555)  OOF AUC: 0.73960
[XGBoost] 시드 8/10 (random_state=999)  OOF AUC: 0.73979
[XGBoost] 시드 9/10 (random_state=31)  OOF AUC: 0.73990
[XGBoost] 시드 10/10 (random_state=77)  OOF AUC: 0.73985

XGBoost 배깅 OOF AUC: 0.74010
저장 완료: oof_xgboost_bagged.npy, test_xgboost_bagged.npy


## 5. 3종(LightGBM+CatBoost+XGBoost, 전부 배깅) 최종 앙상블

In [10]:
oof_lgbm_bagged = np.load(f"{CACHE_DIR}/oof_10seed_bagged.npy")
score_lgbm_bagged = roc_auc_score(y, oof_lgbm_bagged)

print(f"LightGBM(배깅): {score_lgbm_bagged:.5f}")
print(f"CatBoost(배깅): {score_catboost_bagged:.5f}")
print(f"XGBoost(배깅): {score_xgboost_bagged:.5f}")
print()

best_combo, best_score = None, max(score_lgbm_bagged, score_catboost_bagged, score_xgboost_bagged)
step = 0.1
weights = [round(i * step, 1) for i in range(int(1 / step) + 1)]
for w1 in weights:
    for w2 in weights:
        w3 = round(1 - w1 - w2, 2)
        if w3 < 0 or w3 > 1:
            continue
        blend = w1 * oof_lgbm_bagged + w2 * oof_catboost_bagged + w3 * oof_xgboost_bagged
        blend_score = roc_auc_score(y, blend)
        if blend_score > best_score:
            best_score = blend_score
            best_combo = (w1, w2, w3)

print(f"최적 조합: LightGBM {best_combo[0]} + CatBoost {best_combo[1]} + XGBoost {best_combo[2]}")
print(f"점수: {best_score:.5f}")
print()
print(f"참고: 배깅 전 우리 3종 앙상블은 0.74027, 김영혜님 v5는 0.74043(OOF)/0.7417(실제)이었습니다.")

LightGBM(배깅): 0.74036
CatBoost(배깅): 0.74005
XGBoost(배깅): 0.74010

최적 조합: LightGBM 0.5 + CatBoost 0.3 + XGBoost 0.2
점수: 0.74053

참고: 배깅 전 우리 3종 앙상블은 0.74027, 김영혜님 v5는 0.74043(OOF)/0.7417(실제)이었습니다.


## 6. 최종 제출 파일 생성

In [11]:
test_lgbm_bagged = np.load(f"{CACHE_DIR}/test_lgbm_bagged.npy") if os.path.exists(f"{CACHE_DIR}/test_lgbm_bagged.npy") else None

if test_lgbm_bagged is None:
    print("경고: test_lgbm_bagged.npy가 없습니다.")
    print("lgbm_multiseed_bagging.ipynb의 5번 셀에 다음 줄을 추가해서 다시 실행해주세요:")
    print('  np.save(f"{CACHE_DIR}/test_lgbm_bagged.npy", test_pred_bagged)')
else:
    w1, w2, w3 = best_combo
    final_test_pred = w1 * test_lgbm_bagged + w2 * test_catboost_bagged + w3 * test_xgboost_bagged

    submission = pd.DataFrame({"ID": test_ids, "probability": final_test_pred})
    output_path = "1차_3model_bagged_ensemble_submit.csv"
    submission.to_csv(output_path, index=False)
    print(f"저장 완료: {output_path}")
    print(submission.head())

저장 완료: 1차_3model_bagged_ensemble_submit.csv
           ID  probability
0  TEST_00000     0.002226
1  TEST_00001     0.005613
2  TEST_00002     0.153065
3  TEST_00003     0.107468
4  TEST_00004     0.503323


In [12]:
# 7. 우리 배깅 앙상블 + 김영혜님 v5 앙상블 최종 블렌딩 체크
v5_dir = "../팀원파일/김영혜"
v5_ensemble_oof = np.load(f"{v5_dir}/v5_ensemble_oof.npy")
v5_y = np.load(f"{v5_dir}/v5_y.npy")

# y 정합성 확인
n_diff = np.sum(y.astype(int) != v5_y.astype(int))
print(f"y 정합성 확인: 차이나는 행 {n_diff}건" + (" -> 일치!" if n_diff == 0 else " -> 경고: 불일치 있음, 해석 주의"))

w1, w2, w3 = best_combo
our_bagged_ensemble = w1 * oof_lgbm_bagged + w2 * oof_catboost_bagged + w3 * oof_xgboost_bagged

score_ours = roc_auc_score(y, our_bagged_ensemble)
score_v5 = roc_auc_score(y, v5_ensemble_oof)
corr = np.corrcoef(our_bagged_ensemble, v5_ensemble_oof)[0, 1]

print(f"\n우리 배깅 3종 앙상블: {score_ours:.5f}")
print(f"김영혜님 v5 앙상블: {score_v5:.5f}")
print(f"상관계수: {corr:.4f}")
print()

baseline = max(score_ours, score_v5)
best_w_final, best_score_final = (1.0 if score_ours >= score_v5 else 0.0), baseline
for w in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    blend = w * our_bagged_ensemble + (1 - w) * v5_ensemble_oof
    blend_score = roc_auc_score(y, blend)
    marker = " <-- 현재 최고" if blend_score > best_score_final else ""
    if blend_score > best_score_final:
        best_w_final, best_score_final = w, blend_score
    print(f"우리팀 {w:.1f} + v5 {1-w:.1f}: {blend_score:.5f}{marker}")

print(f"\n최종 최적 가중치: 우리팀 {best_w_final:.1f} + v5 {1-best_w_final:.1f}")
print(f"점수: {best_score_final:.5f}  (단독 최고 대비 개선: {best_score_final - baseline:+.5f})")

y 정합성 확인: 차이나는 행 0건 -> 일치!

우리 배깅 3종 앙상블: 0.74053
김영혜님 v5 앙상블: 0.74043
상관계수: 0.9893

우리팀 0.0 + v5 1.0: 0.74043
우리팀 0.1 + v5 0.9: 0.74051
우리팀 0.2 + v5 0.8: 0.74058 <-- 현재 최고
우리팀 0.3 + v5 0.7: 0.74064 <-- 현재 최고
우리팀 0.4 + v5 0.6: 0.74068 <-- 현재 최고
우리팀 0.5 + v5 0.5: 0.74070 <-- 현재 최고
우리팀 0.6 + v5 0.4: 0.74071 <-- 현재 최고
우리팀 0.7 + v5 0.3: 0.74070
우리팀 0.8 + v5 0.2: 0.74066
우리팀 0.9 + v5 0.1: 0.74061
우리팀 1.0 + v5 0.0: 0.74053

최종 최적 가중치: 우리팀 0.6 + v5 0.4
점수: 0.74071  (단독 최고 대비 개선: +0.00018)


In [13]:
# 8. (최적 조합이 의미 있다면) 최종 제출 파일까지 생성
v5_submission = pd.read_csv(f"{v5_dir}/submission_v5_imputed.csv")

# ID 기준으로 안전하게 병합 (행 순서가 다를 수 있으니)
our_test_df = pd.DataFrame({"ID": test_ids, "ours": final_test_pred})
merged = our_test_df.merge(v5_submission.rename(columns={"probability": "v5"}), on="ID", how="inner")
print(f"병합된 행 수: {len(merged)} (원래 test 행 수: {len(test_ids)}와 같아야 정상)")

final_blend = best_w_final * merged["ours"] + (1 - best_w_final) * merged["v5"]
final_submission = pd.DataFrame({"ID": merged["ID"], "probability": final_blend})

output_path = "1차_final_team_blend_submit.csv"
final_submission.to_csv(output_path, index=False)
print(f"\n저장 완료: {output_path}")
print(final_submission.head())

병합된 행 수: 90067 (원래 test 행 수: 90067와 같아야 정상)

저장 완료: 1차_final_team_blend_submit.csv
           ID  probability
0  TEST_00000     0.002398
1  TEST_00001     0.005989
2  TEST_00002     0.190418
3  TEST_00003     0.137858
4  TEST_00004     0.558609
